# Precious metals capstone workflow
This notebook orchestrates the same functions as the CLI. Inputs and outputs are local. Add verified COT release evidence before running.

# Precious metals capstone workflow
This notebook orchestrates the same functions as the CLI. Inputs and outputs are local. Add verified COT release evidence before running.

In [11]:
from pathlib import Path
import os

project_root = next((
    path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / "pyproject.toml").is_file() and (path / "config/config.yml").is_file()
), None)
if project_root is None:
    raise FileNotFoundError("Could not find the project root (pyproject.toml and config/config.yml) from the notebook working directory.")
os.chdir(project_root)

from capstone.cli import read_config, execute
from capstone.data import refresh_sources
config = read_config("config/config.yml", "config/settings.yml")
config

{'data_dir': 'data',
 'release_manifest': 'config/release_dates.csv',
 'start': '2018-01-01',
 'end': None,
 'target_symbols': ['GLD', 'SLV'],
 'required_symbols': ['GLD', 'SLV', 'BTC-USD', 'NVDA'],
 'symbols': {'GLD': 'GLD',
  'SLV': 'SLV',
  'BTC-USD': 'BTC-USD',
  'SOL-USD': 'SOL-USD',
  'ETH-USD': 'ETH-USD',
  'ZEC-USD': 'ZEC-USD',
  'NVDA': 'NVDA',
  '^GSPC': '^GSPC',
  '^IXIC': '^IXIC',
  '^DJI': '^DJI'},
 'market_features': {'BTC-USD': 'btc',
  'SOL-USD': 'sol',
  'ETH-USD': 'eth',
  'ZEC-USD': 'zec',
  'NVDA': 'nvda',
  '^GSPC': 'sp500',
  '^IXIC': 'nasdaq',
  '^DJI': 'dow'},
 'cot_max_age_days': 7,
 'holding_sessions': 5,
 'cross_market_max_age_days': 4,
 'market_download_timeout_seconds': 30,
 'cftc_download_timeout_seconds': 45,
 'train_fraction': 0.6,
 'validation_fraction': 0.2,
 'random_state': 42,
 'ridge_alphas': [1.0, 10.0],
 'tree_depths': [3, 5],
 'tree_min_samples_leaf': 20,
 'forest_estimators': 80,
 'forest_depths': [3, 5],
 'forest_min_samples_leaf': 10,
 'thresh

## Optional refresh
Run this cell when network access is available. It refreshes local snapshots and can overwrite existing files. Alternatively, run `make data` from the project root. CFTC release evidence must be maintained independently.

In [12]:
# refresh_sources(config)  # Uncomment to download; existing snapshots will be refreshed.

## Offline workflow
This calls the same functions as the command line: load verified snapshots, derive features, select models, simulate and report. It requires local snapshots and their hash metadata under `data/`; it does not download missing inputs. If the next cell reports a missing snapshot, run the refresh cell above or `make data` when source access is available, then rerun this cell.

In [14]:
from IPython.display import Markdown, display

try:
    execute(config, refresh=False, stage="run")
except RuntimeError as exc:
    message = str(exc)
    if message.startswith("Required market data ") and "unavailable: Missing cached data or hash metadata:" in message:
        display(Markdown(
            "### Workflow not run: required market snapshot is missing\n\n"
            f"`{message}`\n\n"
            "Run the optional refresh cell when network access is available, then rerun this cell. "
            "No features, model metrics, or backtest results were produced."
        ))
    else:
        print(11)
        raise
except FileNotFoundError as exc:
    message = str(exc)
    if "Missing cached data or hash metadata:" in message:
        display(Markdown(
            "### Workflow not run: required source snapshot is missing\n\n"
            f"`{message}`\n\n"
            "Run the optional refresh cell when network access is available, then rerun this cell. "
            "No features, model metrics, or backtest results were produced."
        ))
    else:
        print(22)
        raise

### Workflow not run: required market snapshot is missing

`Required market data GLD unavailable: Missing cached data or hash metadata: data/raw/market/GLD.csv; run with refresh enabled`

Run the optional refresh cell when network access is available, then rerun this cell. No features, model metrics, or backtest results were produced.